# FAISS vs. ChromaDB — A RAG-Oriented Tour

Two of the most common building blocks for retrieval in RAG pipelines:

| | **FAISS** | **ChromaDB** |
|---|---|---|
| Type | Library (no server, no DB) | Embedded vector database |
| Storage | In-memory + manual `read/write_index` | Persistent (SQLite + parquet) |
| Metadata | **Not built-in** — you maintain a parallel mapping | **Native** — `metadatas=[…]` per chunk |
| Filters | DIY (filter ids before search) | `where={…}` clause, post- or pre-filter |
| Indexes | Flat / IVF / HNSW / PQ / OPQ / … | HNSW only (one knob set) |
| Best for | Throughput, research, custom indexing | Quick RAG, metadata-heavy retrieval |

We'll cover, for each: **(1) general use, (2) similarity metrics, (3) efficiency, (4) metadata for RAG**.

## Setup

In [ ]:
# Optional install — uncomment if needed
# %pip install -q faiss-cpu chromadb sentence-transformers datasets numpy

In [1]:
import os, time, json
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

np.random.seed(0)

# A small slice of arxiv chunks keeps things fast and reproducible.
ds = load_dataset("jamescalam/ai-arxiv-chunked", split="train")
N = 2000
rows = ds.select(range(N))
texts = [r["chunk"] for r in rows]

# Build per-chunk metadata we'll re-use in both backends.
metadatas = []
for i, r in enumerate(rows):
    metadatas.append({
        "chunk_id": i,
        "doc_id": r.get("doi", r.get("id", str(i))),
        "title":   r.get("title", "")[:120],
        "source":  r.get("source", "arxiv"),
        "category": r.get("primary_category", "unknown"),
    })

print(f"Loaded {len(texts)} chunks")
print("Example metadata:", metadatas[0])

Loaded 2000 chunks
Example metadata: {'chunk_id': 0, 'doc_id': '1910.01108', 'title': 'DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter', 'source': 'http://arxiv.org/pdf/1910.01108', 'category': 'cs.CL'}


In [3]:
ds[0]

{'doi': '1910.01108',
 'chunk-id': '0',
 'chunk': 'DistilBERT, a distilled version of BERT: smaller,\nfaster, cheaper and lighter\nVictor SANH, Lysandre DEBUT, Julien CHAUMOND, Thomas WOLF\nHugging Face\n{victor,lysandre,julien,thomas}@huggingface.co\nAbstract\nAs Transfer Learning from large-scale pre-trained models becomes more prevalent\nin Natural Language Processing (NLP), operating these large models in on-theedge and/or under constrained computational training or inference budgets remains\nchallenging. In this work, we propose a method to pre-train a smaller generalpurpose language representation model, called DistilBERT, which can then be ﬁnetuned with good performances on a wide range of tasks like its larger counterparts.\nWhile most prior work investigated the use of distillation for building task-speciﬁc\nmodels, we leverage knowledge distillation during the pre-training phase and show\nthat it is possible to reduce the size of a BERT model by 40%, while retaining 97%\nof i

In [6]:
# arrange data:
data = ds.map(lambda x: {
    "id": f'{x["id"]}-{x["chunk-id"]}',
    "text": x["chunk"],
    "metadata": {
        "title": x["title"],
        "url": x["source"],
        "primary_category": x["primary_category"],
        "published": x["published"],
        "updated": x["updated"],
        "text": x["chunk"],
    }
})

# drop uneeded columns
data = data.remove_columns([
    "title", "summary", "source",
    "authors", "categories", "comment",
    "journal_ref", "primary_category",
    "published", "updated", "references",
    "doi", "chunk-id",
    "chunk"
])
data
data[0]


{'id': '1910.01108-0',
 'text': 'DistilBERT, a distilled version of BERT: smaller,\nfaster, cheaper and lighter\nVictor SANH, Lysandre DEBUT, Julien CHAUMOND, Thomas WOLF\nHugging Face\n{victor,lysandre,julien,thomas}@huggingface.co\nAbstract\nAs Transfer Learning from large-scale pre-trained models becomes more prevalent\nin Natural Language Processing (NLP), operating these large models in on-theedge and/or under constrained computational training or inference budgets remains\nchallenging. In this work, we propose a method to pre-train a smaller generalpurpose language representation model, called DistilBERT, which can then be ﬁnetuned with good performances on a wide range of tasks like its larger counterparts.\nWhile most prior work investigated the use of distillation for building task-speciﬁc\nmodels, we leverage knowledge distillation during the pre-training phase and show\nthat it is possible to reduce the size of a BERT model by 40%, while retaining 97%\nof its language unders

In [ ]:
# Open-source embedding model — 384-d, MIT-licensed, runs on CPU.
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")

def embed(docs, normalize=False):
    v = embedder.encode(
        docs,
        batch_size=64,
        normalize_embeddings=normalize,
        show_progress_bar=False,
    )
    return np.asarray(v, dtype="float32")

embeddings = embed(texts)
embeddings_unit = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
dim = embeddings.shape[1]
print("Embeddings:", embeddings.shape, "dim =", dim)

---
# Section 1 — FAISS

FAISS (Facebook AI Similarity Search) is a **library**, not a database. You hand it a matrix of float32 vectors, it gives you an index object you can `.search()`. It is the workhorse behind a huge fraction of production RAG systems — used directly, or wrapped by higher-level tools.

Key mental model:
- An index is just numbers + an algorithm.
- Ids are integers you assign — **FAISS does not store text or metadata for you**.
- Persistence is `faiss.write_index(idx, path)` / `faiss.read_index(path)`.

## 1.1 General use — IndexFlatL2

In [ ]:
import faiss

index_l2 = faiss.IndexFlatL2(dim)        # exact, brute-force, L2 distance
index_l2.add(embeddings)
print("ntotal:", index_l2.ntotal)

query = "How does retrieval-augmented generation reduce hallucinations?"
q = embed([query])

D, I = index_l2.search(q, k=5)           # D: distances, I: indices
for rank, (idx, dist) in enumerate(zip(I[0], D[0])):
    print(f"#{rank+1}  d={dist:.3f}  {metadatas[idx]['title'][:80]}")

## 1.2 Similarity metrics in FAISS

FAISS gives you **two primitive metrics**:

| Metric | Index class | Higher = closer? | Use it for |
|---|---|---|---|
| L2 (Euclidean²) | `IndexFlatL2` | No — *smaller* is closer | Raw geometric distance |
| Inner product | `IndexFlatIP` | Yes | Dot product / **cosine** (after normalizing) |

**Cosine similarity is not a separate metric** — you get it by L2-normalizing your vectors and using `IndexFlatIP`. After normalization, `dot(a,b) == cos(a,b)`.

In [ ]:
# Cosine via inner product on unit-normalized vectors.
index_cos = faiss.IndexFlatIP(dim)
index_cos.add(embeddings_unit)

q_unit = q / np.linalg.norm(q, axis=1, keepdims=True)
D, I = index_cos.search(q_unit, k=5)
for rank, (idx, sim) in enumerate(zip(I[0], D[0])):
    print(f"#{rank+1}  cos={sim:.3f}  {metadatas[idx]['title'][:80]}")

**Which metric should I pick?**
- Most modern embedding models (OpenAI, BGE, E5, Cohere) are trained with **cosine** in mind → use `IndexFlatIP` + normalized vectors.
- If your vectors carry magnitude meaning (e.g. word counts, raw activations), L2 may be more appropriate.
- L2 and cosine give *identical rankings* on unit-normalized vectors — only the scale differs.

## 1.3 Efficiency — Flat vs. IVF vs. HNSW

`IndexFlat*` is exact: O(N·d) per query. Fine up to ~1M vectors. Beyond that, switch to an **approximate** index that trades a few % of recall for orders of magnitude latency.

In [ ]:
def time_search(index, queries, k=5, repeats=3):
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        D, I = index.search(queries, k)
        times.append((time.perf_counter() - t0) * 1000)
    return min(times), I

# Build three indexes on the same data.
flat = faiss.IndexFlatIP(dim)
flat.add(embeddings_unit)

# IVFFlat: partition into `nlist` cells (k-means), search the closest `nprobe`.
nlist = 64
quantizer = faiss.IndexFlatIP(dim)
ivf = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
ivf.train(embeddings_unit)
ivf.add(embeddings_unit)
ivf.nprobe = 8                                  # search 8/64 cells

# HNSW: multi-layer graph, skip-list-like.
hnsw = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efConstruction = 80
hnsw.hnsw.efSearch = 32
hnsw.add(embeddings_unit)

queries = embed([
    "transformer attention complexity",
    "diffusion models for image generation",
    "reinforcement learning reward shaping",
    "prompt engineering chain of thought",
    "vector database ANN benchmarks",
], normalize=True)

t_flat, I_flat = time_search(flat, queries)
t_ivf,  I_ivf  = time_search(ivf,  queries)
t_hnsw, I_hnsw = time_search(hnsw, queries)

def recall(approx, truth, k=5):
    hits = sum(len(set(a) & set(t)) for a, t in zip(approx, truth))
    return hits / (len(truth) * k)

print(f"Flat   : {t_flat:6.2f} ms   recall@5 = 1.000  (exact)")
print(f"IVFFlat: {t_ivf:6.2f} ms   recall@5 = {recall(I_ivf,  I_flat):.3f}")
print(f"HNSW   : {t_hnsw:6.2f} ms   recall@5 = {recall(I_hnsw, I_flat):.3f}")

**Tuning knobs to remember:**
- **IVF** — `nlist` (build, ~√N), `nprobe` (per query). More probes ⇒ higher recall, slower.
- **HNSW** — `M` (build, neighbours per node), `efConstruction` (build quality), `efSearch` (per query, the recall/latency dial).
- The right operating point is where you hit your **recall target** (e.g. 0.95) at the **lowest p95 latency**.

## 1.4 Metadata for RAG — the FAISS pattern

FAISS stores **only vectors and integer ids**. You keep everything else in a parallel structure.

Standard RAG pattern:
1. Build a list/array of `metadatas` aligned by id.
2. Search FAISS → get back ids.
3. Look up text + metadata by id and feed to the LLM.

For pre-filtering (e.g. "only category = cs.CL"), you build an `IDSelectorBatch` or run a separate filtered index.

In [ ]:
def faiss_retrieve(query, k=4, category=None):
    qv = embed([query], normalize=True)

    if category is None:
        D, I = flat.search(qv, k)
        ids, scores = I[0], D[0]
    else:
        # Pre-filter: build an IDSelector restricted to matching ids.
        allowed = np.array(
            [m["chunk_id"] for m in metadatas if m["category"] == category],
            dtype="int64",
        )
        sel = faiss.IDSelectorBatch(allowed)
        params = faiss.SearchParameters(sel=sel)
        D, I = flat.search(qv, k, params=params)
        ids, scores = I[0], D[0]

    hits = []
    for idx, sc in zip(ids, scores):
        if idx == -1:
            continue
        hits.append({
            "score": float(sc),
            "text":  texts[idx][:200],
            "meta":  metadatas[idx],
        })
    return hits

# Unfiltered
for h in faiss_retrieve("contrastive learning of sentence embeddings", k=3):
    print(f"[{h['meta']['category']}] {h['meta']['title'][:80]}  ({h['score']:.3f})")

In [ ]:
# Filtered to a single category — DIY pre-filter via IDSelector.
common_cat = max({m["category"] for m in metadatas}, key=lambda c: sum(m["category"] == c for m in metadatas))
print("Filtering to category =", common_cat)
for h in faiss_retrieve("transformer language model", k=3, category=common_cat):
    print(f"[{h['meta']['category']}] {h['meta']['title'][:80]}  ({h['score']:.3f})")

In [ ]:
# Persistence — small but important for production.
faiss.write_index(flat, "/tmp/faiss_flat.index")
loaded = faiss.read_index("/tmp/faiss_flat.index")
print("Reloaded ntotal:", loaded.ntotal)
# Note: you must persist `metadatas` and `texts` yourself (json, parquet, sqlite…).

## 1.5 Metadata the easy way — the LangChain FAISS wrapper

The "FAISS has no metadata" limitation in §1.4 is true of **raw** FAISS. In
practice most RAG code does not touch raw FAISS — it uses
`langchain_community.vectorstores.FAISS`, which wraps the same index with:

- an **`InMemoryDocstore`** + an `index_to_docstore_id` map → every vector now
  carries its `page_content` **and** `metadata`, exactly like a Chroma row;
- a **`filter=`** argument on `similarity_search*` (dict-equality *or* a callable);
- one-call persistence: **`save_local()` / `load_local()`** for vectors **and**
  text **and** metadata together.

Under the hood it is *still* a `faiss.Index` (`IndexFlatIP` here, but you can
hand it IVF/HNSW/PQ). So you keep FAISS speed and index choice while the
wrapper does the bookkeeping §1.4 made you do by hand.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.embeddings import Embeddings

class STEmbeddings(Embeddings):
    # Adapter so the notebook's existing embed() powers a LangChain store.
    def embed_documents(self, docs):
        return embed(list(docs), normalize=True).tolist()
    def embed_query(self, text):
        return embed([text], normalize=True)[0].tolist()

# Reuse the embeddings we already computed (unit vectors ⇒ inner product = cosine).
lc_faiss = FAISS.from_embeddings(
    text_embeddings=list(zip(texts, embeddings_unit.tolist())),
    embedding=STEmbeddings(),
    metadatas=metadatas,
    distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT,
)
print("underlying index :", type(lc_faiss.index).__name__,
      "| ntotal:", lc_faiss.index.ntotal)
print("docstore entries :", len(lc_faiss.index_to_docstore_id),
      "(id → text + metadata, handled for you)")

In [ ]:
# Same query as §1.1 — but text + metadata come back attached to each hit.
q = "How does retrieval-augmented generation reduce hallucinations?"
for doc, score in lc_faiss.similarity_search_with_score(q, k=5):
    print(f"cos={score:.3f}  [{doc.metadata['category']}]  "
          f"{doc.metadata['title'][:75]}")

**Metadata filtering — the thing §1.4 needed an `IDSelector` for:**

`filter=` accepts a dict (equality) or a callable. One important difference
from Chroma: the LangChain FAISS `filter` is a **post-filter** applied to the
top `fetch_k` candidates (default `fetch_k=20`). For a selective filter, raise
`fetch_k` so enough survivors remain — Chroma's `where=` is a true pre-filter.

In [ ]:
common_cat = max({m["category"] for m in metadatas},
                  key=lambda c: sum(m["category"] == c for m in metadatas))
print("filter: category =", common_cat)

# dict-equality filter
hits = lc_faiss.similarity_search(
    "transformer language model", k=3,
    filter={"category": common_cat},
    fetch_k=300,                       # widen pool: filter is a POST-filter
)
for d in hits:
    print(f"[{d.metadata['category']}]  {d.metadata['title'][:75]}")

# callable filter (arbitrary predicate over metadata) also works
recent = lc_faiss.similarity_search(
    "language model evaluation", k=3,
    filter=lambda m: m["category"] == common_cat,
    fetch_k=300,
)
print("callable filter →", [d.metadata["category"] for d in recent])

In [ ]:
# One call persists vectors + text + metadata together
# (contrast §1.4, where you json/parquet the metadata yourself).
lc_faiss.save_local("/tmp/lc_faiss_store")
reloaded = FAISS.load_local(
    "/tmp/lc_faiss_store", STEmbeddings(),
    allow_dangerous_deserialization=True,   # it unpickles the docstore
)
print("reloaded vectors :", reloaded.index.ntotal)
print("reloaded docs    :", len(reloaded.index_to_docstore_id))
doc0 = reloaded.docstore.search(reloaded.index_to_docstore_id[0])
print("metadata survived:", doc0.metadata)

In [ ]:
# It is still FAISS: the index is a real faiss.Index, so every FAISS lever
# (IVF / HNSW / PQ / OPQ) is still available — pass a prebuilt index via
# FAISS(embedding_function=..., index=<faiss index>, docstore=..., ...).
import faiss
print("type(lc_faiss.index) :", type(lc_faiss.index))
print("is a faiss.Index     :", isinstance(lc_faiss.index, faiss.Index))
print("→ wrapper owns id→(text, metadata); FAISS still owns the math.")

**FAISS takeaways for RAG:**
- ✅ Fastest, most flexible vector library; rich choice of indexes (Flat / IVF / HNSW / PQ / IVFPQ / OPQ).
- ✅ Multiple metrics (L2, IP); cosine via normalization.
- ⚠️ You own metadata storage, persistence, and filtering — fine for ~10k LOC RAG systems, painful for prototypes.
- ⚠️ No update/delete semantics for most indexes (Flat supports `remove_ids`; HNSW/IVF generally don't).
- ➕ **LangChain `FAISS` wrapper** (§1.5) bolts on the metadata docstore, `filter=`, and one-call `save_local/load_local` — closing most of the gap with Chroma while keeping raw-FAISS speed and index choice. Caveat: its `filter` is a **post-filter** over `fetch_k` candidates, not a pre-filter.

---
# Section 2 — ChromaDB

Chroma is an **embedded vector database**: think SQLite for vectors. One pip install, no server, persistence is on by default. Each "collection" stores `(id, embedding, document, metadata)` rows and is backed by an HNSW index.

## 2.1 General use — create, add, query

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="/tmp/chroma_demo")
# Drop if it exists so the demo is reproducible.
if any(c.name == "arxiv_demo" for c in client.list_collections()):
    client.delete_collection("arxiv_demo")

col = client.create_collection(
    name="arxiv_demo",
    metadata={"hnsw:space": "cosine"},   # similarity metric — see §2.2
)

col.add(
    ids=[str(i) for i in range(len(texts))],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadatas,
)
print("count:", col.count())

In [ ]:
res = col.query(
    query_embeddings=embed(["how does retrieval-augmented generation work"]).tolist(),
    n_results=5,
)
for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
    print(f"d={dist:.3f}  [{meta['category']}] {meta['title'][:80]}")

## 2.2 Similarity metrics in Chroma

Chroma exposes the metric through `metadata={"hnsw:space": ...}` at collection creation:

| `hnsw:space` | Meaning | Distance returned |
|---|---|---|
| `"l2"` (default) | Squared Euclidean | smaller = closer |
| `"ip"` | Inner product | smaller = closer (note: it's `1 - dot` style) |
| `"cosine"` | Cosine distance | `1 - cos_sim`; smaller = closer |

**Important:** Chroma always returns *distances* (smaller = better), even for cosine. If you want a similarity score, compute `sim = 1 - distance`.

Unlike FAISS, the metric is locked when the collection is created — to switch you create a new collection.

In [ ]:
# Quick demo: same data, three metrics, same top-1?
for space in ["l2", "ip", "cosine"]:
    name = f"demo_{space}"
    if any(c.name == name for c in client.list_collections()):
        client.delete_collection(name)
    c = client.create_collection(name=name, metadata={"hnsw:space": space})
    c.add(
        ids=[str(i) for i in range(200)],
        embeddings=embeddings[:200].tolist(),
        documents=texts[:200],
        metadatas=metadatas[:200],
    )
    r = c.query(query_embeddings=embed(["attention is all you need"]).tolist(), n_results=1)
    print(f"{space:7s}  d={r['distances'][0][0]:.4f}  {r['metadatas'][0][0]['title'][:70]}")

## 2.3 Efficiency — Chroma's HNSW knobs

Chroma uses HNSW under the hood. The same three knobs FAISS exposes are available via collection metadata:

| Key | Meaning | Default | Effect |
|---|---|---|---|
| `hnsw:M` | neighbours per node | 16 | larger = better recall, slower build, more RAM |
| `hnsw:construction_ef` | candidate-list size at build | 100 | larger = better recall, slower build |
| `hnsw:search_ef` | candidate-list size at query | 100 | larger = better recall, slower search |

Tuning loop is the same as FAISS-HNSW: keep raising `search_ef` until recall@k against a ground-truth (FAISS Flat) crosses your target.

In [ ]:
# Build a Chroma collection with custom HNSW params.
if any(c.name == "arxiv_tuned" for c in client.list_collections()):
    client.delete_collection("arxiv_tuned")

tuned = client.create_collection(
    name="arxiv_tuned",
    metadata={
        "hnsw:space": "cosine",
        "hnsw:M": 32,
        "hnsw:construction_ef": 200,
        "hnsw:search_ef": 64,
    },
)
tuned.add(
    ids=[str(i) for i in range(len(texts))],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadatas,
)

# Compare timing vs the default collection.
qv_list = embed([
    "transformer attention complexity",
    "diffusion models for image generation",
    "reinforcement learning reward shaping",
]).tolist()

for label, c in [("default", col), ("tuned", tuned)]:
    t0 = time.perf_counter()
    c.query(query_embeddings=qv_list, n_results=5)
    print(f"{label:8s}: {(time.perf_counter()-t0)*1000:6.2f} ms")

## 2.4 Metadata for RAG — Chroma's killer feature

Chroma stores `metadatas` natively and supports a JSON-like filter language on them. Filtering is a first-class part of `.query()`, not something you bolt on.

Operators: `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`, combined with `$and` / `$or`.

Two filter slots:
- `where=...` — filter on metadata fields.
- `where_document=...` — full-text contains/regex on the chunk text itself.

In [ ]:
# Filter by category (single value)
common_cat = max({m["category"] for m in metadatas}, key=lambda c: sum(m["category"] == c for m in metadatas))

res = col.query(
    query_embeddings=embed(["language model evaluation"]).tolist(),
    n_results=3,
    where={"category": common_cat},
)
for meta, dist in zip(res["metadatas"][0], res["distances"][0]):
    print(f"[{meta['category']}] d={dist:.3f}  {meta['title'][:80]}")

In [ ]:
# Combined filter: category in a set AND title contains 'attention'
categories_top = list({m["category"] for m in metadatas})[:3]
res = col.query(
    query_embeddings=embed(["self-attention mechanisms"]).tolist(),
    n_results=5,
    where={"category": {"$in": categories_top}},
    where_document={"$contains": "attention"},
)
for meta, dist, doc in zip(res["metadatas"][0], res["distances"][0], res["documents"][0]):
    print(f"[{meta['category']}] d={dist:.3f}  {meta['title'][:70]}")
    print(f"   …{doc[:120].strip()}…\n")

In [ ]:
# A tiny end-to-end RAG retrieval helper.
def rag_retrieve(query, k=4, where=None):
    qv = embed([query]).tolist()
    res = col.query(query_embeddings=qv, n_results=k, where=where)
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({
            "score": 1 - dist,            # distance → similarity for cosine
            "text":  doc,
            "source": f"{meta['title']} ({meta['doc_id']})",
        })
    return hits

context = rag_retrieve("how does RAG ground LLM answers?", k=3)
for h in context:
    print(f"sim={h['score']:.3f}  {h['source'][:90]}")

# This is what you'd splice into a prompt:
prompt_context = "\n\n".join(f"[{i+1}] {h['text'][:300]}" for i, h in enumerate(context))
print("\n--- prompt context (truncated) ---")
print(prompt_context[:600], "…")

**Chroma takeaways for RAG:**
- ✅ One object, four pieces (`ids`, `embeddings`, `documents`, `metadatas`) — the natural shape of a RAG chunk.
- ✅ Native filtering with `where` and `where_document` — perfect for tenant-scoped, time-bounded, or topic-restricted retrieval.
- ✅ Persistence and updates/deletes are first-class.
- ⚠️ Only HNSW; no PQ/IVF for memory compression.
- ⚠️ Local/embedded — needs a server (Chroma Cloud, or run the docker image) for multi-process access.

---
# Section 3 — Side-by-side cheat-sheet

| Concern | FAISS | Chroma |
|---|---|---|
| **Setup** | `pip install faiss-cpu`, build matrix | `pip install chromadb`, one client call |
| **Metric** | L2 / IP (+ cosine via normalization) | `hnsw:space` ∈ {l2, ip, cosine} at create-time |
| **Indexes** | Flat / IVF / HNSW / PQ / OPQ … | HNSW only |
| **Tuning** | per-index params (`nprobe`, `efSearch`, …) | `hnsw:M`, `construction_ef`, `search_ef` |
| **Metadata** | external mapping, DIY persistence | native column, JSON filter language |
| **Filters** | `IDSelector` pre-filter, manual | `where=…`, `where_document=…` |
| **Updates** | Flat: yes; HNSW/IVF: rebuild | `add`, `update`, `upsert`, `delete` |
| **Persistence** | `read/write_index` for vectors only | automatic on the client path |
| **Best fit** | hot retrieval path, billions of vectors, custom recipes | course/prototype/RAG with rich metadata up to a few M chunks |

**Rule of thumb:** start with **Chroma** when building a RAG demo; reach for **FAISS** when you outgrow Chroma's HNSW-only footprint, need PQ-style compression, or want to stitch retrieval into a custom pipeline.

> **On the metadata / persistence rows:** those limitations describe *raw* FAISS. Via `langchain_community.vectorstores.FAISS` (§1.5) you get a native-feeling `(id, embedding, text, metadata)` docstore, `filter=` queries, and one-call persistence. The genuine remaining trade-offs are **index variety & raw speed** (FAISS) vs **first-class updates/deletes, server mode, and true pre-filtering** (Chroma).